# Phase 1: EDA & MSTL Baseline

1. Load the DataCo dataset
2. Turn daily transactions into weekly demand series
3. Clean the dataset by handling missing values and removing outliers
4. Exploratory Data Analysis (trends, seasonality, distribution)
5. MSTL forecast
6. MAE, RMSE, sMAPE

## Setup

In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))

from src.config import DATA_RAW, DATAFILE
from src.utils.data_loader import DataLoader
from src.utils.data_cleaner import DataCleaner
from src.eda.exploratory_analysis import ExploratoryAnalysis
from src.forecasting.mstl_model import MSTLModel
from src.utils.visualization import (
    plot_time_series,
    plot_seasonality,
    plot_autocorrelation,
    plot_monthly_distribution,
    plot_forecast_comparison,
)

print('Setup complete.')

Setup complete.


## Load Data

In [ ]:
loader = DataLoader(DATA_RAW / DATAFILE)
raw_df = loader.load()

summary = loader.get_summary()
print(f"Rows × Columns   : {summary['shape']}")
print(f"Date range       : {summary['date_range'][0].date()} - {summary['date_range'][1].date()}")
print(f"Duplicates dropped: {summary['duplicate_rows_dropped']}")
print()

# Show columns with missing values
missing = {k: v for k, v in summary['missing_values'].items() if v > 0}
print('Columns with missing values:')
for col, count in missing.items():
    print(f'  {col:<40} {count:>6} ({100*count/summary["shape"][0]:.1f}%)')

Rows × Columns   : (180519, 53)
Date range       : 2015-01-01 → 2018-01-31
Duplicates dropped: 0

Columns with missing values:
  Customer Lname                                8 (0.0%)
  Customer Zipcode                              3 (0.0%)
  Order Zipcode                            155679 (86.2%)
  Product Description                      180519 (100.0%)


## Clean Data

In [ ]:
cleaner = DataCleaner(raw_df)

# each stage of cleaning
weekly_raw    = cleaner.aggregate_by_frequency()
weekly_filled = cleaner.handle_missing_values()
weekly_clean  = cleaner.remove_outliers()

print(f"Weekly periods   : {len(weekly_clean)}")
print(f"Demand range     : {weekly_clean.min():.0f} – {weekly_clean.max():.0f} units/week")
print(f"Mean / Std       : {weekly_clean.mean():.1f} / {weekly_clean.std():.1f}")
print()

# before and after
comparison = pd.DataFrame({
    'before': weekly_raw.describe().round(1),
    'after' : weekly_clean.describe().round(1),
})
print('before vs after:')
print(comparison.to_string())

Weekly periods   : 162
Demand range     : 2239 – 2906 units/week
Mean / Std       : 2571.9 / 160.0

before vs after:
       before   after
count   162.0   162.0
mean   2370.9  2571.9
std     688.0   160.0
min     205.0  2239.2
25%    2506.2  2506.2
50%    2609.5  2609.5
75%    2684.2  2684.2
max    2906.0  2906.0


## Train / Test Split

In [ ]:
train, test = cleaner.train_test_split()

print(f"Train : {len(train)} weeks  ({train.index[0].date()} - {train.index[-1].date()})")
print(f"Test  : {len(test)} weeks  ({test.index[0].date()}  - {test.index[-1].date()})")
print()
print('Test set actuals:')
for date, val in test.items():
    print(f'  {date.date()}  -  {val:.0f} units')

Train : 158 weeks  (2015-01-04 - 2018-01-07)
Test  : 4 weeks  (2018-01-14  - 2018-02-04)

Test set actuals:
  2018-01-14  -  2239 units
  2018-01-21  -  2239 units
  2018-01-28  -  2239 units
  2018-02-04  -  2239 units


## EDA

In [ ]:
eda = ExploratoryAnalysis(weekly_clean)
trend_info = eda.analyze_trends()

print(f"Trend direction  : {trend_info['direction']}")
print(f"Slope            : {trend_info['slope']:.2f} units/week")
print(f"R^2              : {trend_info['r_squared']:.3f}")
print(f"p-value          : {trend_info['p_value']:.4f}")
print(f"Total change     : {trend_info['pct_change']:+.1f}%  ({trend_info['first_value']:.0f} - {trend_info['last_value']:.0f})")
print(f"Stationary       : {trend_info['is_stationary']}")

fig = plot_time_series(weekly_clean, title='Full History (cleaned)')
fig.show()

Trend direction  : downward
Slope            : -1.98 units/week
R^2              : 0.336
p-value          : 0.0000
Total change     : +0.0%  (2239 - 2239)
Stationary       : False


## EDA: Seasonality

In [ ]:
season_info = eda.detect_seasonality(period=52)

print(f"Seasonal strength : {season_info['seasonal_strength']:.3f}")
print(f"ACF at lag 52     : {season_info['acf_at_period']:.3f}")
print(f"Seasonality detected: {season_info['has_seasonality']}")

fig_decomp = plot_seasonality(weekly_clean, period=52)
fig_decomp.show()

fig_acf = plot_autocorrelation(weekly_clean, lags=60)
fig_acf.show()

Seasonal strength : 0.166
ACF at lag 52     : -0.099
Seasonality detected: False


## EDA: Distribution

In [ ]:
dist_info = eda.get_distribution_stats()

print('Distribution summary:')
for key in ('mean', 'median', 'std', 'min', 'max', 'skewness', 'kurtosis', 'cv'):
    print(f'  {key:<12} {dist_info[key]:>10.3f}')
print(f'  {"outliers":<12} {dist_info["outlier_count"]:>10}')

print()
print('Monthly mean demand:')
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
for m, mean_val in dist_info['monthly_means'].items():
    print(f'  {month_names[m-1]}  {mean_val:>7.0f}')

fig_monthly = plot_monthly_distribution(weekly_clean)
fig_monthly.show()

Distribution summary:
  mean           2571.943
  median         2609.500
  std             160.037
  min            2239.250
  max            2906.000
  skewness         -0.821
  kurtosis          0.021
  cv                0.062
  outliers              0

Monthly mean demand:
  Jan     2527
  Feb     2621
  Mar     2631
  Apr     2657
  May     2576
  Jun     2532
  Jul     2594
  Aug     2579
  Sep     2606
  Oct     2517
  Nov     2537
  Dec     2505


## MSTL Forecast

In [ ]:
model = MSTLModel(horizon=4, seasonal_periods=[52, 4])
model.fit(train)

forecast = model.predict()

print('Forecast vs Actual:')
print(f"  {'Date':<14} {'Forecast':>11} {'Actual':>9} {'Error':>10}")
print('  ' + '-' * 46)
for date, fc_val, act_val in zip(forecast.index, forecast.values, test.values):
    error = fc_val - act_val
    print(f'  {str(date.date()):<14} {fc_val:>10.1f} {act_val:>10.1f} {error:>+10.1f}')

Forecast vs Actual:
  Date              Forecast    Actual      Error
  ----------------------------------------------
  2018-01-14         2309.7     2239.2      +70.5
  2018-01-21         2365.5     2239.2     +126.3
  2018-01-28         2291.8     2239.2      +52.6
  2018-02-04         2390.4     2239.2     +151.2


## Evaluation

In [ ]:
metrics = model.evaluate(test, forecast)

print('MSTL Model — Phase 1 Results')
print('=' * 35)
print(f"  MAE    : {metrics['mae']:.2f} units")
print(f"  RMSE   : {metrics['rmse']:.2f} units")
print(f"  sMAPE  : {metrics['smape']:.2f}%")
print()
print('Note: sMAPE > 50% is expected for MSTL on this dataset.')
print('Phase 2 (N-BEATS) targets ~27% sMAPE.')

fig_fc = plot_forecast_comparison(
    actual=test,
    forecast=forecast,
    title=f'MSTL Forecast vs Actual  |  MAE={metrics["mae"]:.0f}  sMAPE={metrics["smape"]:.1f}%',
    train=train,
)
fig_fc.show()

MSTL Model — Phase 1 Results
  MAE    : 100.14 units
  RMSE   : 107.87 units
  sMAPE  : 4.36%

Note: sMAPE > 50% is expected for MSTL on this dataset.
Phase 2 (N-BEATS) targets ~27% sMAPE.
